In [13]:
import numpy as np
from sklearn.decomposition import PCA
import pandas as pd
import scanpy as sc
import anndata as ad
import scipy as sp
import matplotlib.pylab as pl
import matplotlib.pyplot as plt
import seaborn as sns
import random
from speciesot_helpers import label_uniform_cell_type_row, \
                              top_n_organisms_from_species, \
                              cell_types_with_n_per_organism, \
                              sample_equal_cell_types, \
                              ot_between_organisms, \
                              calc_random_mean_std, \
                              test_train_split_adata, \
                              get_cell_type_from_index, \
                              create_label_to_sample_dicts, \
                              concat_index, \
                              show_cell_type_results, \
                              cell_type_knn_acc
from pytorch_helpers import fit_mlp
import perturbot
from perturbot.match import get_coupling_egw_labels_ott
from perturbot.predict import train_mlp

In [2]:
human_dir = 'data/tabula_sapiens/'
mouse_dir = 'data/tabula_muris/'

In [3]:
human_adatas = top_n_organisms_from_species(human_dir, 8, 'human')

In [4]:
mouse_adatas = top_n_organisms_from_species(mouse_dir, 8, 'mouse')

In [5]:
all_adatas = mouse_adatas + human_adatas
pairs = [(i,i+8) for i in range(8)]

In [6]:
# Gather data
n_samples = 1500
a1_full, a2_full = sample_equal_cell_types(all_adatas[2], all_adatas[9], n_samples)

In [7]:
# remove T cells that aren't in t_cell_keep_list
t_cell_keep_list = ['CD4-positive, alpha-beta T cell','CD8-positive, alpha-beta T cell','regulatory T cell']

In [8]:
a1_t_cells = a1_full[a1_full.obs['shared_cell_type']=='T cell']

In [9]:
a1_keep_t_cells = a1_full[a1_full.obs['cell_type'].isin(t_cell_keep_list)]

In [10]:
a2_t_cells = a2_full[a2_full.obs['shared_cell_type']=='T cell']

In [11]:
a2_keep_t_cells = a2_full[a2_full.obs['cell_type'].isin(t_cell_keep_list)]

In [14]:
a1_all_subtypes_keep_list, a2_all_subtypes_keep_list = [], []

for t_cell_subtype in t_cell_keep_list:
    a1_subtype = a1_full[a1_full.obs['cell_type']==t_cell_subtype]
    a2_subtype = a2_full[a2_full.obs['cell_type']==t_cell_subtype]
    subtype_cell_num = np.min([len(a1_subtype), len(a2_subtype)])
    a1_subtype_keep = a1_subtype[:subtype_cell_num]
    a2_subtype_keep = a2_subtype[:subtype_cell_num]
    a1_all_subtypes_keep_list.append(a1_subtype_keep)
    a2_all_subtypes_keep_list.append(a2_subtype_keep)

a1_subtypes_keep = ad.concat(a1_all_subtypes_keep_list)
a2_subtypes_keep = ad.concat(a2_all_subtypes_keep_list)

In [15]:
a1_non_t_cells = a1_full[~(a1_full.obs['shared_cell_type']=='T cell')]

In [16]:
a2_non_t_cells = a2_full[~(a2_full.obs['shared_cell_type']=='T cell')]

In [17]:
a1_full_for_ot = ad.concat([a1_non_t_cells, a1_subtypes_keep])

In [18]:
a2_full_for_ot = ad.concat([a2_non_t_cells, a2_subtypes_keep])

In [19]:
X_dict, Y_dict, X_full_dict, Y_full_dict, X_index_dict, Y_index_dict = create_label_to_sample_dicts(a1_full_for_ot, a2_full_for_ot)

In [20]:
# Learn matching in the latent space
T_dict, log = get_coupling_egw_labels_ott((X_dict, Y_dict), 5e-6) 

running EGWL with ott
GW called
lse step
updating linearization
Label considered for Sinkhorn run
lse step
updating linearization
Label considered for Sinkhorn run
5 outer iterations were needed.
The last Sinkhorn iteration has converged: False
The outer loop of Gromov Wasserstein has converged: False
The final regularized GW cost is: -inf
Done running LEGWOT with ott


In [21]:
num_t_cells = int(a2_full_for_ot.obs['shared_cell_type'].value_counts()['T cell'])

In [22]:
t_cell_idx = None
for ct_num in T_dict:
    if len(T_dict[ct_num]) == num_t_cells:
        t_cell_idx = int(ct_num)

In [23]:
t_cell_T = T_dict[t_cell_idx]

In [31]:
t_cell_indices = Y_index_dict[t_cell_idx]

In [28]:
t_cell_T

array([[2.0702380e-13, 0.0000000e+00, 4.2571941e-29, ..., 6.6786354e-34,
        0.0000000e+00, 8.5081440e-36],
       [2.0918438e-38, 0.0000000e+00, 1.6255733e-37, ..., 8.6553274e-13,
        0.0000000e+00, 2.3781682e-30],
       [0.0000000e+00, 0.0000000e+00, 3.4944803e-06, ..., 9.2038830e-14,
        0.0000000e+00, 3.5344819e-38],
       ...,
       [0.0000000e+00, 0.0000000e+00, 1.9526983e-33, ..., 1.4762328e-30,
        0.0000000e+00, 0.0000000e+00],
       [4.7386152e-16, 1.3541872e-22, 0.0000000e+00, ..., 0.0000000e+00,
        0.0000000e+00, 1.0601382e-25],
       [0.0000000e+00, 0.0000000e+00, 0.0000000e+00, ..., 0.0000000e+00,
        0.0000000e+00, 0.0000000e+00]], shape=(62, 62), dtype=float32)

In [24]:
pairings = np.argmax(t_cell_T, axis=1)

In [39]:
a1_t_cell_ordered = a1_full_for_ot[X_index_dict[t_cell_idx]].obs['cell_type']

In [35]:
a2_t_cell_ordered = a2_full_for_ot[t_cell_indices[pairings]].obs['cell_type']

In [47]:
a1_t_cell_ordered.tail(50)

index
AATCCAGCATCGATGT-1-53-0-0    CD4-positive, alpha-beta T cell
GTTACAGGTCGTCTTC-1-53-0-0    CD4-positive, alpha-beta T cell
AGGGTGAAGAAACCTA-1-53-0-0    CD4-positive, alpha-beta T cell
CACCTTGTCCTGTACC-1-53-0-0    CD4-positive, alpha-beta T cell
CATCAAGTCATCGATG-1-53-0-0    CD4-positive, alpha-beta T cell
ACAGCCGCAGCCTTTC-1-53-0-0    CD4-positive, alpha-beta T cell
ATTACTCCATTAGCCA-1-53-0-0    CD4-positive, alpha-beta T cell
GACGTGCCAAATTGCC-1-53-0-0    CD4-positive, alpha-beta T cell
CGCGTTTCACTTAACG-1-53-0-0    CD4-positive, alpha-beta T cell
GTCCTCACAGTTTACG-1-53-0-0    CD4-positive, alpha-beta T cell
TTTGTCACATCACGTA-1-53-0-0    CD4-positive, alpha-beta T cell
GGCAATTTCCATGAAC-1-53-0-0    CD4-positive, alpha-beta T cell
TCGTACCGTTAAGGGC-1-53-0-0    CD8-positive, alpha-beta T cell
AGCTCCTGTTCGGGCT-1-53-0-0    CD8-positive, alpha-beta T cell
ATAGACCAGCGCCTCA-1-53-0-0    CD8-positive, alpha-beta T cell
CGTTAGAGTGCGATAG-1-53-0-0    CD8-positive, alpha-beta T cell
TCTTCGGCAAGTTAAG-1

In [48]:
a2_t_cell_ordered.tail(50)

TSP2_LymphNode_NA_10X_2_1_ATCAGGTAGATAACGT             CD4-positive, alpha-beta T cell
TSP2_LymphNode_NA_10X_2_1_TGGGTTACAACACACT             CD8-positive, alpha-beta T cell
TSP2_LymphNode_NA_10X_2_1_GACTTCCAGATGAACT             CD4-positive, alpha-beta T cell
TSP2_LymphNode_NA_10X_1_1_TTCATTGCATTCCTAT             CD4-positive, alpha-beta T cell
TSP2_Blood_NA_10X_2_1_GGGACCTTCCCTCTTT                 CD4-positive, alpha-beta T cell
TSP2_Bladder_NA_10X_1_1_TGCCGAGTCTGCTTAT                             regulatory T cell
TSP2_Blood_NA_10X_1_1_SheelaPrep_GAGTGTTAGCGTCAAG      CD4-positive, alpha-beta T cell
TSP2_Kidney_NA_10X_1_2_TTACTGTGTCTAGGTT                CD4-positive, alpha-beta T cell
TSP2_Lung_proxmedialdistal_10X_1_2_CTCAACCGTCGAATTC    CD4-positive, alpha-beta T cell
TSP2_BM_vertebralbody_SS2_B113698_B133341_Immune_E2    CD4-positive, alpha-beta T cell
TSP2_Thymus_NA_10X_1_2_AGTGCCGAGAAGCGAA                CD4-positive, alpha-beta T cell
TSP2_LymphNode_NA_10X_1_1_GTCGTTCTCCCAGTGG 

In [ ]:
# visualize pairings

In [25]:
Y_paired_dict = {}
for i in range(len(T_dict)):
    pairings_i = np.argmax(T_dict[i], axis=1)
    Y_paired_dict[i] = np.zeros(Y_dict[i].shape)
    for j in range(len(pairings_i)):
        Y_paired_dict[i][j] = Y_dict[i][pairings_i[j]]

X_paired_stacked = np.vstack([X_dict[k] for k in X_dict])  
Y_paired_stacked = np.vstack([Y_paired_dict[k] for k in Y_paired_dict])

In [26]:
donornum = 2
all_adatas[donornum][all_adatas[donornum].obs['shared_cell_type']=='T cell'].obs['cell_type'].value_counts()

NameError: name 'donorn3m' is not defined

In [ ]:
donornum = 9
all_adatas[donornum][all_adatas[donornum].obs['shared_cell_type']=='T cell'].obs['cell_type'].value_counts()

In [ ]:
all_adatas[donornum][all_adatas[donornum].obs['cell_type'].isin(['CD4-positive, alpha-beta T cell','CD8-positive, alpha-beta T cell','regulatory T cell'])].obs['cell_type']

In [ ]:
X_index_dict[0]